In [ ]:
!pip install smolagents -U ddgs markdownify requests

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.colab.output import eval_js
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, Tool, tool, DuckDuckGoSearchTool, VisitWebpageTool, FinalAnswerTool

In [ ]:
from smolagents import tool

@tool
def get_games(date: str) -> list:
    """Fetch NBA games and scores from ESPN for a date.

    Args:
        date: The scoreboard date in YYYYMMDD format, e.g. 20261106.
    """
    import requests

    url = (
        "https://site.api.espn.com/apis/site/v2/"
        "sports/basketball/nba/scoreboard"
    )
    response = requests.get(
        url,
        params={"dates": date},
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()

    games = []

    for event in data.get("events", []):
        competition = event["competitions"][0]

        games.append({
            "game_id": event["id"],
            "matchup": event["name"],
            "start_time_utc": event["date"],
            "completed": competition["status"]["type"]["completed"],
            "teams": [
                {
                    "name": team["team"]["displayName"],
                    "score": team.get("score"),
                }
                for team in competition["competitors"]
            ],
        })

    return games

In [ ]:
games = get_games("20261112")

for game in games:
    print(game)

In [ ]:
@tool
def get_game_details(game_id: str) -> dict:
    """Fetch leading scorers, player stats, and headlines for an NBA game.

    Args:
        game_id: A real ESPN game ID returned by get_games.
    """
    import requests

    url = (
        "https://site.api.espn.com/apis/site/v2/"
        "sports/basketball/nba/summary"
    )
    response = requests.get(
        url,
        params={"event": game_id},
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()

    teams = []

    for team in data.get("boxscore", {}).get("players", []):
        players = []

        for group in team.get("statistics", []):
            labels = group.get("labels", [])

            for player in group.get("athletes", []):
                if player.get("didNotPlay"):
                    continue

                stats = dict(zip(labels, player.get("stats", [])))
                players.append({
                    "name": player["athlete"]["displayName"],
                    "stats": stats,
                })

        players.sort(
            key=lambda player: float(
                player["stats"].get("PTS", "0") or "0"
            ),
            reverse=True,
        )

        teams.append({
            "team": team["team"]["displayName"],
            "top_scorers": players[:3],
        })

    return {
        "game_id": game_id,
        "teams": teams,
        "source_url": (
            f"https://www.espn.com/nba/game/_/gameId/{game_id}"
        ),
    }

In [ ]:
print(get_game_details("401909976"))

In [ ]:
user_timezone = ZoneInfo(
    eval_js("Intl.DateTimeFormat().resolvedOptions().timeZone")
)

@tool
def get_today_date() -> str:
    """Returns today's date in the user's timezone in YYYYMMDD format."""
    return datetime.now(user_timezone).strftime("%Y%m%d")

In [ ]:
NBA_INSTRUCTIONS = """
You are an NBA game analyst. Use only facts returned by your tools.
Answer the user's specific question. Treat a general request for an NBA
update as a full-slate recap.

Dates and game lookup
- For "today" or "yesterday", call get_today_date() first. It should
  return the user's local NBA date in YYYYMMDD format. For "yesterday",
  subtract one calendar day from that date. Use explicit dates as given.
- The NBA date rolls over at 5:00 a.m. in the user's local timezone.
  At 10:00 p.m. on Nov 12 and at 2:00 or 4:00 a.m. on Nov 13,
  the NBA date is Nov 12. At 5:00 a.m. on Nov 13, it becomes Nov 13.
- Call get_games(date=...) with the resolved date in YYYYMMDD format.
  Use the returned slate even if an evening game's UTC start timestamp
  shows the next calendar date. Reuse successful lookups for identical
  tool calls.

What to investigate
- Full-slate update: call get_game_details() for every game returned.
  Analyze the most interesting verified highlights for each matchup.
- Best game to watch: compare completed games, fetch details for the
  strongest candidates, and recommend one with two or three concrete
  reasons. Honor spoiler-free requests.
- Specific matchup: fetch details for that game and focus on what
  the user asked about.
- If no game matches, state which date and matchup you checked.

What to report
- Final: give the score and explain what made the game worth watching,
  using standout performances or verified turning points. If the game
  was one-sided, focus on the standout performance if there was one.
- Live: give the current score, period, and clock when available.
  Highlight the leading players in key categories such as points,
  rebounds, and assists when the data supports it. Prioritize standout
  performances over a list of category leaders.
- Upcoming: give the scheduled tipoff if available.
- Claim overtime, comebacks, lead changes, or dramatic finishes only
  when the tools explicitly support them. A close final score supports
  a close margin. Minutes played alone cannot establish overtime.
- If a detail is missing or a lookup fails, use the verified facts
  you have and mention the gap if it affects the answer.

Output
- Pass a readable Markdown string to final_answer().
- Give each game its own matchup heading in a multi-game update.
- Include only the most interesting stats. Keep each game brief.
- Keep code, JSON, and raw tool output out of the final answer.
"""

In [ ]:
model = InferenceClientModel(
    model_id="nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4",
    provider="baseten",
)

agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        FinalAnswerTool(),
        get_games,
        get_game_details,
        get_today_date
    ],
    model=model,
    max_steps=10,
    verbosity_level=2,
    instructions=NBA_INSTRUCTIONS
)


question = input("Ask an NBA question: ")
print(agent.run(question))